# FastLane small models: QLoRA on Qwen2.5-3B (Kaggle, both adapters in one run)

Trains up to **two LoRA adapters** back to back on one T4 (set `TRAIN` in the next cell):
- `all_domains` → the FastLane small model (used by FastLane and most variants), ~3.3 h
- `movies_buses` → the new-domain test adapter for variant A8 (no restaurant data), ~2.3 h

**Setup:** upload `data/train/colab_bundle.zip` (from `fastlane train colab-bundle`) as a new version of the
**fastlane-train** dataset. In this notebook: *Add Input* → fastlane-train (latest version), *Accelerator* → GPU T4 x2,
*Internet* → On. Then *Save Version* → **Save & Run All (Commit)** and close the tab.

**Result:** in the version's *Output* tab, download `fastlane-qwen2.5-3b-adapter.zip` and `fastlane-qwen2.5-3b-mb-adapter.zip`.
On your machine unzip them into `kaggle_output/` and run
`fastlane train finish kaggle_output/fastlane-qwen2.5-3b/adapter --name fastlane-qwen2.5-3b` (and the same with `-mb`).

In [ ]:
MODEL = "Qwen/Qwen2.5-3B-Instruct"
ADAPTERS = {"all_domains": "fastlane-qwen2.5-3b", "movies_buses": "fastlane-qwen2.5-3b-mb"}
# Which adapters this run trains. Data v3 only changed A2 steps, which the movies+buses adapter (A8) never
# sees in use, so v3 retrains all_domains only. Add "movies_buses" to retrain both.
TRAIN = ["all_domains"]
EPOCHS = 2

In [ ]:
!pip -q install -U "transformers>=4.45" "peft>=0.12" "trl>=0.20" "datasets>=2.20" "bitsandbytes>=0.43" accelerate
# Kaggle ships an old torchao that breaks peft; it is not needed.
!pip -q uninstall -y torchao
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import glob, os, subprocess
script = glob.glob("/kaggle/input/**/train_lora.py", recursive=True)[0]
data_root = os.path.dirname(os.path.dirname(glob.glob("/kaggle/input/**/all_domains/train.jsonl", recursive=True)[0]))
print(script, data_root)
for a in TRAIN:
    print(a, sum(1 for _ in open(f"{data_root}/{a}/train.jsonl")), "train /", sum(1 for _ in open(f"{data_root}/{a}/val.jsonl")), "val")

In [ ]:
# Train each adapter (loss on the target JSON only) and zip it straight away, so finished adapters are kept
# in the Output tab even if a later run fails. check=True stops the notebook on an error.
for adapter in TRAIN:
    name = ADAPTERS[adapter]
    out = f"/kaggle/working/{name}"
    subprocess.run(["python", script, "--data", f"{data_root}/{adapter}", "--model", MODEL, "--out", out,
                    "--epochs", str(EPOCHS), "--batch", "2", "--grad-accum", "8"],
                   check=True, env={**os.environ, "CUDA_VISIBLE_DEVICES": "0"})
    subprocess.run(f"cd /kaggle/working && zip -q -r {name}-adapter.zip {name}/adapter {name}/eval.json && rm -rf {name}/checkpoint-*",
                   shell=True, check=True)
    print(name, open(f"{out}/eval.json").read())

In [ ]:
!ls -lh /kaggle/working